# Stage 2 · RLHF Pipeline — EXERCISES
### Topics: Bradley-Terry · Reward Model Training · KL-Constrained RL · PPO-for-LLMs · Reward Hacking

> Fill every `# TODO`. Run `# ASSERT` cells to verify.  
> This is the bridge between raw RL (Stage 1) and modern methods (Stages 3–5).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass


---
## 1 · Bradley-Terry Preference Model

### Why not just collect scalar reward labels?
Human labellers are inconsistent at giving absolute scores, but reliable at **pairwise comparisons**:  
*"Which of these two completions is better?"*

### Bradley-Terry model
Given two completions $y_w$ (winner) and $y_l$ (loser), the probability of the human preferring $y_w$ is:

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

where $r(x, y)$ is a scalar reward and $\sigma$ is the sigmoid.

### Reward model training loss (negative log-likelihood)
$$\mathcal{L}_{RM} = -\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma(r(x, y_w) - r(x, y_l))\right]$$

**Key properties:**
- Only the *difference* in rewards matters, not absolute scale
- Equivalent to binary cross-entropy where label is always "chosen > rejected"
- Reward model is initialised from the SFT checkpoint (same backbone as the policy)


In [ ]:
def bradley_terry_loss(
    reward_chosen:   torch.Tensor,  # (B,)
    reward_rejected: torch.Tensor,  # (B,)
) -> torch.Tensor:
    """
    -mean( log σ(r_chosen - r_rejected) )
    Use F.logsigmoid for numerical stability (avoids log(sigmoid(x)) precision loss).
    """
    # TODO
    raise NotImplementedError


def reward_model_accuracy(
    reward_chosen:   torch.Tensor,
    reward_rejected: torch.Tensor,
) -> float:
    """Fraction of pairs where r_chosen > r_rejected. No gradient needed."""
    # TODO
    raise NotImplementedError


class RewardModel(nn.Module):
    """MLP reward model: cat(prompt_emb, completion_emb) → scalar reward."""
    def __init__(self, embed_dim: int = 32, hidden: int = 64):
        super().__init__()
        # TODO: 2-hidden-layer MLP with ReLU, input=embed_dim*2, output=1
        raise NotImplementedError

    def forward(self, prompt_emb: torch.Tensor, completion_emb: torch.Tensor) -> torch.Tensor:
        # TODO: concat, pass through net, squeeze last dim → (B,)
        raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, D = 8, 32
r_eq = torch.zeros(B)
loss_eq = bradley_terry_loss(r_eq, r_eq)
assert abs(loss_eq.item() - math.log(2)) < 1e-5, f"Equal rewards should give log(2), got {loss_eq.item()}"

r_c_big = torch.full((B,), 10.0)
r_r_big = torch.full((B,), -10.0)
assert bradley_terry_loss(r_c_big, r_r_big).item() < 1e-4

rm = RewardModel(D)
r_c = rm(torch.randn(B, D), torch.randn(B, D))
r_r = rm(torch.randn(B, D), torch.randn(B, D))
assert r_c.shape == (B,)
print(f"bradley_terry_loss ✓  RewardModel ✓  accuracy={reward_model_accuracy(r_c, r_r):.2f}")


---
## 2 · Reward Model Training

The reward model is trained on a dataset of preference pairs $(x, y_w, y_l)$.

### Dataset construction
1. Collect prompts from users
2. Generate multiple completions per prompt using the SFT model
3. Have human labellers rank pairs (or use AI feedback)
4. Create binary preference pairs from rankings

### Training details (from InstructGPT)
- Initialise from SFT model (same backbone, add a linear scalar head)
- Shuffle all pairs; train with BT loss
- Use held-out accuracy as early stopping criterion
- **Important:** treat each comparison independently (not per-prompt)

### Preference dataset simulation
For testing, we simulate a "true" reward function and generate synthetic preferences:
$y_w \succ y_l \iff r_{true}(y_w) > r_{true}(y_l)$  (with occasional noise)


In [ ]:
@dataclass
class PreferencePair:
    prompt_emb:   torch.Tensor
    chosen_emb:   torch.Tensor
    rejected_emb: torch.Tensor


def make_synthetic_dataset(
    n_pairs: int = 200,
    embed_dim: int = 32,
    noise: float = 0.1,
    seed: int = 0,
) -> List[PreferencePair]:
    """
    Generate synthetic preference pairs.
    True reward = dot(completion_emb, fixed_direction).
    With probability `noise`, flip the chosen/rejected label (simulates human noise).
    """
    # TODO:
    #  1. Fix a random unit-norm direction (the "true" reward direction)
    #  2. For each pair: sample prompt_emb, emb_a, emb_b
    #  3. Assign chosen = higher dot-product with direction
    #  4. Flip with probability noise
    raise NotImplementedError


def train_reward_model(
    rm: RewardModel,
    dataset: List[PreferencePair],
    n_epochs: int = 10,
    batch_size: int = 32,
    lr: float = 1e-3,
) -> List[float]:
    """
    Train RM with BT loss. Return per-epoch accuracy list.
    Steps per epoch:
      - shuffle dataset
      - for each minibatch: stack tensors, forward rm, compute BT loss, step
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(1)
dataset = make_synthetic_dataset(n_pairs=500, embed_dim=32, noise=0.05)
rm      = RewardModel(embed_dim=32)
history = train_reward_model(rm, dataset, n_epochs=15, batch_size=64, lr=3e-3)
assert history[-1] > 0.80,        f"Expected >80% acc, got {history[-1]:.3f}"
assert history[-1] > history[0],   "Accuracy should improve"
print(f"RM training ✓  acc: {history[0]:.3f} → {history[-1]:.3f}")


---
## 3 · KL-Constrained RL — The Three-Model Setup

### InstructGPT architecture
```
SFT Model  (π_ref)  — frozen reference; initialisation of policy
Policy     (π_θ)    — trained via RL; starts as copy of π_ref
Reward     (r_φ)    — frozen after RM training; scores completions
```

### The full RLHF objective
$$\max_\theta \mathbb{E}_{x \sim D,\, y \sim \pi_\theta(\cdot|x)}\left[r_\phi(x,y) - \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}\right]$$

The KL term $\beta \cdot D_{KL}(\pi_\theta \| \pi_{ref})$ is **crucial**:
1. Prevents reward hacking — without it, the policy finds degenerate completions the RM rates highly
2. Preserves language quality — keeps the model readable and on-distribution
3. Acts as a regulariser — $\beta$ is the most important hyperparameter

### Effective reward = RM reward − KL penalty
$$r_{eff}(x,y) = r_\phi(x,y) - \beta \cdot \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

The policy optimises $r_{eff}$, not just $r_\phi$.


In [ ]:
def kl_divergence_approx(
    logprobs_policy: torch.Tensor,  # (B,)
    logprobs_ref:    torch.Tensor,  # (B,)  detached
) -> torch.Tensor:
    """
    Per-sequence KL approximation (unbiased under π_θ):
      KL ≈ log π_θ(y) - log π_ref(y)
    Returns (B,).
    """
    # TODO
    raise NotImplementedError


def effective_reward(
    rm_reward:       torch.Tensor,  # (B,)
    logprobs_policy: torch.Tensor,  # (B,)  detached (used in reward, not in policy grad directly)
    logprobs_ref:    torch.Tensor,  # (B,)
    beta: float = 0.1,
) -> torch.Tensor:
    """
    r_eff(x,y) = r_φ(x,y) - β * KL(π_θ || π_ref)
    This is the reward signal fed to the advantage computation.
    """
    # TODO
    raise NotImplementedError


def kl_coefficient_schedule(
    step: int,
    target_kl: float,
    current_kl: float,
    beta: float,
    factor: float = 1.5,
) -> float:
    """
    Adaptive beta:
      if current_kl > target_kl * factor → beta *= factor
      if current_kl < target_kl / factor → beta /= factor
      else                               → beta unchanged
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B       = 8
lp_pol  = torch.randn(B)
lp_ref  = torch.randn(B)
rm_r    = torch.rand(B)

assert kl_divergence_approx(lp_ref, lp_ref).abs().max() < 1e-5, "KL(same)=0"
r_eff = effective_reward(rm_r, lp_pol, lp_ref, beta=0.1)
assert r_eff.shape == (B,)

beta_up   = kl_coefficient_schedule(0, 0.01, 0.05, 0.1)
beta_down = kl_coefficient_schedule(0, 0.01, 0.001, 0.1)
assert beta_up > 0.1 and beta_down < 0.1
print("kl_divergence_approx ✓  effective_reward ✓  kl_coefficient_schedule ✓")


---
## 4 · PPO Adapted for LLMs

Standard PPO was designed for low-dimensional action spaces. Applying it to LLMs requires several adaptations.

### Key differences

| Aspect | Standard PPO | PPO for LLMs |
|---|---|---|
| Action space | Small (e.g., 4 actions) | Huge vocabulary (50k+ tokens) |
| Episode length | Hundreds of steps | Tens to hundreds of tokens |
| Reward | Per-step from env | Terminal (end of sequence) from RM |
| Value network | Separate architecture | Value head on same backbone |
| Ratio computation | Per step | Per token, averaged over sequence |

### Token-level vs sequence-level log-probs
The probability ratio is computed per-sequence (sum of per-token log-probs):

$$\rho = \exp\!\left(\sum_t \log \pi_\theta(y_t|x,y_{<t}) - \sum_t \log \pi_{\theta_{old}}(y_t|x,y_{<t})\right)$$

### Reward assignment
Most RLHF implementations assign the scalar RM reward **only to the last token**  
and propagate backwards via GAE. This makes the reward effectively per-sequence.

$$r_t = \begin{cases} r_{eff}(x,y) - \beta \cdot KL_t & t = T \\ -\beta \cdot KL_t & t < T \end{cases}$$


In [ ]:
def compute_per_token_kl(
    logits_policy: torch.Tensor,   # (B, T, V)
    logits_ref:    torch.Tensor,   # (B, T, V)  — detach before use
) -> torch.Tensor:                 # (B, T)  — non-negative
    """
    Full KL(π_θ || π_ref) at each token position.
    KL = Σ_v p(v) * (log p(v) - log q(v))
    Steps:
      log_p = log_softmax(logits_policy, dim=-1)
      log_q = log_softmax(logits_ref.detach(), dim=-1)
      p     = log_p.exp()
      return (p * (log_p - log_q)).sum(dim=-1)
    """
    # TODO
    raise NotImplementedError


def build_token_rewards(
    rm_reward:  torch.Tensor,   # (B,)
    per_tok_kl: torch.Tensor,   # (B, T)  detached
    comp_mask:  torch.Tensor,   # (B, T)
    beta: float = 0.1,
) -> torch.Tensor:              # (B, T)
    """
    Assign rewards to token positions:
      all completion tokens: -beta * KL_t
      last completion token: also add rm_reward[b]
      prompt tokens:         0
    Hint: find last completion token per sequence with (comp_mask * arange).argmax(dim=-1)
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B, T, V = 4, 12, 50
prompt_len = 6
logits_pol = torch.randn(B, T, V)
logits_ref = torch.randn(B, T, V)
comp_mask  = torch.cat([torch.zeros(B,prompt_len,dtype=torch.long),
                        torch.ones(B,T-prompt_len,dtype=torch.long)], dim=1)
rm_reward  = torch.rand(B)

kl_tok = compute_per_token_kl(logits_pol, logits_ref)
assert kl_tok.shape == (B, T) and (kl_tok >= 0).all()

tok_r = build_token_rewards(rm_reward, kl_tok.detach(), comp_mask, beta=0.1)
assert (tok_r[:, :prompt_len] == 0).all(), "Prompt rewards must be 0"
print(f"compute_per_token_kl ✓  build_token_rewards ✓")
print(f"  Last token reward (b=0): {tok_r[0,11].item():.4f}")


---
## 5 · Reward Hacking, Goodhart's Law & Training Diagnostics

> *"When a measure becomes a target, it ceases to be a good measure."* — Goodhart's Law

### What is reward hacking?
The RM is an imperfect proxy for human preferences. Without the KL penalty,  
the policy finds **out-of-distribution outputs** that fool the RM while being low quality:
- Repeating tokens / phrases the RM rates highly
- Generating very long or very short outputs
- Adding sycophantic phrases ("Great question!")
- Drifting to incoherent but high-RM-score text

### Empirical signals of reward hacking

| Metric | Healthy | Hacking signal |
|---|---|---|
| RM reward | Steadily increasing | Sudden spike then plateau |
| KL from ref | Slowly increasing | KL explodes (> 20 nats) |
| Entropy | Gradually decreasing | Collapse to near 0 |
| Human eval | Tracks RM reward | Diverges from RM reward |
| Perplexity under SFT | Slightly increasing | Large increase |

### Mitigation strategies
1. **KL penalty (β):** primary defence; adaptive schedule helps
2. **Reward model ensembles:** use min of ensemble predictions (pessimistic reward)
3. **RM score clipping:** `r_eff = min(r_φ, r_max)` prevents extrapolation
4. **EOS token trick:** stop generation if policy produces EOS early
5. **Iterative RM updates:** collect new human labels on RL-generated outputs and retrain RM


In [ ]:
def monitor_training(
    logprobs_policy: torch.Tensor,  # (B,)
    logprobs_ref:    torch.Tensor,  # (B,)
    rm_rewards:      torch.Tensor,  # (B,)
    entropy:         torch.Tensor,  # scalar
    step:            int,
    beta:            float,
) -> Dict[str, float]:
    """
    Compute: kl = mean(logp_pol - logp_ref)
             effective_rew = mean(rm_rewards) - beta * kl
    Return dict with keys: step, rm_reward, kl_from_ref, effective_rew, entropy, beta.
    """
    # TODO
    raise NotImplementedError


def detect_reward_hacking(
    metrics_history: List[Dict[str, float]],
    kl_threshold:    float = 10.0,
    entropy_min:     float = 0.1,
    rm_spike_factor: float = 2.0,
) -> List[str]:
    """
    Return list of warning strings if any of:
      - kl_from_ref > kl_threshold
      - entropy < entropy_min
      - mean rm_reward in last 5 steps > rm_spike_factor * mean rm_reward in first 5 steps
    Empty list = healthy.
    """
    # TODO
    raise NotImplementedError


# ── ASSERT ────────────────────────────────────────────────────────────────
torch.manual_seed(0)
B       = 16
lp_pol  = torch.randn(B)
lp_ref  = torch.randn(B)
rm_rew  = torch.rand(B) * 2
entropy = torch.tensor(0.8)

metrics = monitor_training(lp_pol, lp_ref, rm_rew, entropy, step=100, beta=0.1)
assert "rm_reward" in metrics and "kl_from_ref" in metrics and "entropy" in metrics

history_healthy = [{"rm_reward": 0.5+i*0.01, "kl_from_ref": 0.5, "entropy": 1.0-i*0.01} for i in range(20)]
history_hacking = history_healthy[:10] + [{"rm_reward":5.0,"kl_from_ref":15.0,"entropy":0.02} for _ in range(10)]
assert len(detect_reward_hacking(history_healthy)) == 0, "False alarm on healthy run"
assert len(detect_reward_hacking(history_hacking)) > 0,  "Missed hacking signals"
print("monitor_training ✓  detect_reward_hacking ✓")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
